# 🔬 Demo End-to-End: XAI + AI Agent Pipeline — sample_0000 (Improved Cross-Attention)
## Multimodal Restaurant Review Scoring — CrossAttentionFusion (PhoBERT × Swin-B)

**Nhóm 24 — SE365 | Trình bày: Demo cuối kỳ**

Notebook này trình diễn **đúng một mẫu test cố định**:

```python
SAMPLE_INDEX = 0
SAMPLE_ID = "sample_0000"
CASE_TYPE = "fixed_test_index_0"
```

Mẫu này là dòng đầu tiên (`idx=0`) của `data/text/test.csv`, chứa review về
**bánh canh cua**.

Pipeline đầy đủ cho mẫu này:
1. **Dự đoán** (CrossAttentionFusion) → 5 điểm: Food / Price / Atmosphere / Service / Overall
2. **Grad-CAM** — vùng ảnh quan trọng cho Overall Satisfaction
3. **PhoBERT Attention** — attention gộp ở mức từ, không hiển thị mảnh subword
4. **Cross-Attention (phiên bản cải tiến)** — hai hình trực quan hóa mới, dễ đọc
   cho giảng viên: Token → Patch và Patch → Token
5. **SHAP** — đóng góp text-origin / image-origin sau cross-attention
6. **LIME** — giải thích cục bộ (text + image) cho đúng mẫu đang xem
7. **AI Agent** (GPT-4o) — chỉ chạy khi Colab Secret `OPENAI_API_KEY` tồn tại

> Notebook này **không** thực hiện quét/so sánh nhiều mẫu. Không có "Mẫu A/B/C".
> Toàn bộ artifact được lưu dưới `sample_0000/` trên Google Drive.

---
## 0.2 · Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print('✅ Google Drive mounted at /content/drive')

---
## 0.3 · Clone repo & checkout branch `final_demo`

In [ ]:
# ── 0.3 · Clone repo & checkout branch `final_demo` ─────────────────────────
import os
import shlex
import shutil
import subprocess

REPO_URL     = 'https://github.com/lechihoang/SE365.git'
PROJECT_ROOT = '/content/SE365'
BRANCH       = 'final_15'


def run_cmd(cmd, cwd=None):
    '''Run a command without invoking a shell; raise with captured output.'''
    if not isinstance(cmd, (list, tuple)) or not cmd:
        raise TypeError('cmd must be a non-empty list/tuple of arguments')
    printable = shlex.join([str(part) for part in cmd])
    print(f'$ {printable}')
    result = subprocess.run(
        [str(part) for part in cmd],
        cwd=cwd,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
    )
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        raise RuntimeError(
            f'Command failed ({result.returncode}): {printable}\n'
            f'{result.stdout or ""}'
        )
    return result.stdout or ''


os.chdir('/content')
if os.path.exists(PROJECT_ROOT):
    if not os.path.isdir(PROJECT_ROOT):
        raise RuntimeError(f'PROJECT_ROOT exists but is not a directory: {PROJECT_ROOT}')
    print(f'Removing existing repo: {PROJECT_ROOT}')
    shutil.rmtree(PROJECT_ROOT)

run_cmd([
    'git', 'clone', '--branch', BRANCH, '--single-branch',
    REPO_URL, PROJECT_ROOT,
])

git_dir = os.path.join(PROJECT_ROOT, '.git')
if not os.path.isdir(git_dir):
    raise FileNotFoundError(f'Git repository was not created: {git_dir}')

current_branch = run_cmd(
    ['git', 'branch', '--show-current'], cwd=PROJECT_ROOT
).strip()
if current_branch != BRANCH:
    raise RuntimeError(f'Expected branch {BRANCH!r}, got {current_branch!r}')

os.chdir(PROJECT_ROOT)
print(f'✅ Working directory: {os.getcwd()}')
print(f'✅ Branch: {current_branch}')

---
## 0.4 · Install thư viện bổ sung (nếu thiếu)

In [ ]:
# Install the repository requirements first, then demo-only dependencies.
import importlib.util
import sys

REQUIREMENTS_FILE = os.path.join(PROJECT_ROOT, 'requirements.txt')
if not os.path.isfile(REQUIREMENTS_FILE):
    raise FileNotFoundError(f'Missing requirements file: {REQUIREMENTS_FILE}')

run_cmd([
    sys.executable, '-m', 'pip', 'install', '-q',
    '-r', REQUIREMENTS_FILE,
])

OPTIONAL_PACKAGES = {
    'shap': 'shap',
    'lime': 'lime',
    'seaborn': 'seaborn',
    'skimage': 'scikit-image',
    'dotenv': 'python-dotenv',
}
missing_packages = [
    package
    for module, package in OPTIONAL_PACKAGES.items()
    if importlib.util.find_spec(module) is None
]
if missing_packages:
    run_cmd([
        sys.executable, '-m', 'pip', 'install', '-q',
        *missing_packages,
    ])

# The agent code uses the modern `OpenAI(...)` client API.
run_cmd([
    sys.executable, '-m', 'pip', 'install', '-q', 'openai>=1.0',
])

print('✅ Tất cả thư viện đã sẵn sàng.')

---
## 0.5 · Tải & giải nén dữ liệu (`data.zip`) từ Google Drive

In [ ]:
# ── Giải nén dữ liệu từ Google Drive, không dùng shell command ───────────────
import zipfile

DRIVE_DATA_ZIP = '/content/drive/MyDrive/SE365/data.zip'
DATA_DIR_LOCAL = os.path.join(PROJECT_ROOT, 'data')

if not os.path.isfile(DRIVE_DATA_ZIP):
    raise FileNotFoundError(
        f'Không tìm thấy data.zip trên Drive: {DRIVE_DATA_ZIP}'
    )

if os.path.isdir(DATA_DIR_LOCAL):
    print(f'Removing existing data directory: {DATA_DIR_LOCAL}')
    shutil.rmtree(DATA_DIR_LOCAL)

project_real = os.path.realpath(PROJECT_ROOT)
with zipfile.ZipFile(DRIVE_DATA_ZIP, 'r') as archive:
    members = archive.infolist()
    if not members:
        raise ValueError(f'Archive is empty: {DRIVE_DATA_ZIP}')

    # Reject path traversal before extracting.
    for member in members:
        destination = os.path.realpath(
            os.path.join(PROJECT_ROOT, member.filename)
        )
        if os.path.commonpath([project_real, destination]) != project_real:
            raise ValueError(
                f'Unsafe path in data archive: {member.filename!r}'
            )
    archive.extractall(PROJECT_ROOT)

expected_test_csv = os.path.join(DATA_DIR_LOCAL, 'text', 'test.csv')
expected_image_dir = os.path.join(DATA_DIR_LOCAL, 'image')
if not os.path.isfile(expected_test_csv):
    roots = sorted({
        member.filename.replace('\\', '/').split('/')[0]
        for member in members if member.filename
    })
    raise FileNotFoundError(
        'Giải nén hoàn tất nhưng không tìm thấy data/text/test.csv. '
        f'Các thư mục gốc trong archive: {roots}'
    )
if not os.path.isdir(expected_image_dir):
    raise FileNotFoundError(
        f'Giải nén hoàn tất nhưng thiếu thư mục ảnh: {expected_image_dir}'
    )

print(f'✅ Dữ liệu đã giải nén vào {DATA_DIR_LOCAL}')

---
## 0.6 · Imports

In [ ]:
import json
import textwrap
import traceback
import warnings

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

matplotlib.rcParams['font.family'] = 'DejaVu Sans'
warnings.filterwarnings('ignore')

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from xai.config import (
    TARGET_NAMES, DISPLAY_NAMES, FACTOR_NAMES, LABEL_COLS,
    FUSED_DIM, CROSS_ATTN_HIDDEN_DIM,
    COLOR_SCHEMES, DEFAULT_DPI, THESIS_DPI,
    BEST_EXP_ID, DEFAULT_SEED,
    BEST_TEXT_MODEL, BEST_IMAGE_MODEL,
)
from xai.utils import (
    load_model, get_tokenizer, get_image_processor,
    load_single_sample, get_prediction,
    get_device, set_seed,
)
from xai.gradcam_explainer import (
    compute_gradcam_for_image, overlay_cam_on_image, find_target_layer,
)
from xai.attention_explainer import (
    extract_phobert_attention, aggregate_attention,
    cls_token_importance, merge_subword_attention,
    plot_cls_importance_bar,
    extract_cross_attention,
)
from xai.shap_explainer import (
    FusionHeadWrapper, extract_fused_embeddings,
    select_background, compute_shap_values, modality_contribution,
)
from xai.lime_explainer import (
    run_lime_image, run_lime_text,
    save_lime_image_explanation,
)
from xai.case_study import check_sample_artifacts

try:
    from agent import ExplanationAgent
    from agent.config import AgentConfig
    AGENT_AVAILABLE = True
except Exception as _agent_import_error:
    AGENT_AVAILABLE = False
    print(
        '[WARN] AI Agent import failed; XAI demo will continue without it: '
        f'{type(_agent_import_error).__name__}: {_agent_import_error}'
    )

set_seed(DEFAULT_SEED)
print('✅ Imports hoàn thành.')

---
## 0.7 · Cấu hình đường dẫn & thiết bị

In [ ]:
device = get_device()
print(f'Device: {device}')

DRIVE_ROOT = '/content/drive/MyDrive/SE365'
EXP_ID     = BEST_EXP_ID
EXP_DIR    = os.path.join(DRIVE_ROOT, 'experiments', EXP_ID)
XAI_DIR    = os.path.join(EXP_DIR, 'xai')
DEMO_OUT   = os.path.join(DRIVE_ROOT, 'demo_e2e')
AGENT_DIR  = os.path.join(DEMO_OUT, 'agent_reports')

DATA_DIR  = os.path.join(PROJECT_ROOT, 'data', 'text')
IMAGE_DIR = os.path.join(PROJECT_ROOT, 'data', 'image')
CSV_TEST  = os.path.join(DATA_DIR, 'test.csv')
CHECKPOINT_PATH = os.path.join(EXP_DIR, 'best_model_train_fusion.pth')

# Validate upstream inputs before creating output folders. Creating XAI_DIR
# first would accidentally create a missing EXP_DIR and hide the real error.
required_files = {
    'test CSV': CSV_TEST,
    'model checkpoint': CHECKPOINT_PATH,
}
required_dirs = {
    'experiment directory': EXP_DIR,
    'image cache': IMAGE_DIR,
}
missing_inputs = [
    f'{label}: {path}'
    for label, path in {**required_dirs, **required_files}.items()
    if not (os.path.isdir(path) if label in required_dirs else os.path.isfile(path))
]
if missing_inputs:
    raise FileNotFoundError(
        'Thiếu input bắt buộc cho demo:\n- ' + '\n- '.join(missing_inputs)
    )

for output_dir in [XAI_DIR, DEMO_OUT, AGENT_DIR]:
    os.makedirs(output_dir, exist_ok=True)

print(f'PROJECT_ROOT : {PROJECT_ROOT}')
print(f'DRIVE_ROOT   : {DRIVE_ROOT}')
print(f'EXP_DIR      : {EXP_DIR}')
print(f'XAI_DIR      : {XAI_DIR}')
print(f'DEMO_OUT     : {DEMO_OUT}')
print(f'CSV_TEST     : {CSV_TEST}')
print(f'IMAGE_DIR    : {IMAGE_DIR}')

---
## 0.8 · Tải mô hình, tokenizer, image processor

In [ ]:
print('Đang tải mô hình CrossAttentionFusion...')
model, model_config = load_model(EXP_DIR, device)
model.eval()
print(f'✅ Mô hình: {type(model).__name__}')

text_model_name = model_config.get('text_model_name') or BEST_TEXT_MODEL
image_model_name = model_config.get('image_model_name') or BEST_IMAGE_MODEL
tokenizer = get_tokenizer(text_model_name)
image_processor = get_image_processor(image_model_name)

print(f'✅ Tokenizer: {type(tokenizer).__name__} ({text_model_name})')
print(f'✅ Image processor: {type(image_processor).__name__} ({image_model_name})')

target_layer = find_target_layer(model)
print(f'✅ Grad-CAM target layer: {type(target_layer).__name__}')

# load_model(..., xai_mode=True) already enables eager attention exactly once.
text_encoder = getattr(model.text_model, 'encoder', None)
attn_impl = getattr(
    getattr(text_encoder, 'config', None),
    '_attn_implementation',
    None,
)
print(f'✅ PhoBERT attention implementation: {attn_impl or "configured"}')

---
## 0.9 · Hàm tiện ích demo (một mẫu)

In [ ]:
# ── run_safe: bao bọc mỗi bước XAI ─────────────────────────────────────────

def run_safe(fn, step_name='XAI step', fallback=None, **kwargs):
    try:
        return fn(**kwargs)
    except Exception as e:
        print(f'  [SKIP] {step_name}: {type(e).__name__}: {e}')
        if os.environ.get('XAI_TRACEBACK'):
            traceback.print_exc()
        return fallback

# ── Bảng dự đoán vs thực tế ─────────────────────────────────────────────────

def display_prediction_table(pred_result, sample_id=''):
    preds = pred_result['predictions']
    gt    = pred_result['ground_truth']
    errs  = pred_result['absolute_errors']
    rows  = []
    for name, disp in zip(TARGET_NAMES, DISPLAY_NAMES):
        rows.append({'Chỉ tiêu': disp,
                     'Thực tế' : f'{gt[name]:.1f}',
                     'Dự đoán' : f'{preds[name]:.1f}',
                     'AE'      : f'{errs[name]:.2f}'})
    df = pd.DataFrame(rows)
    mae = pred_result['mean_mae']
    print(f'\n{"="*50}')
    print(f'  {sample_id}  —  MAE = {mae:.3f}')
    print('='*50)
    print(df.to_string(index=False))
    print('='*50)
    return df

# ── Biểu đồ dự đoán ─────────────────────────────────────────────────────────
# NOTE: fixed a duplicate-rendering bug from the source notebook. The original
# `plot_prediction_bars` called `plt.show()` AND returned the `Figure` object;
# when the returned figure was the last expression of a notebook cell, Jupyter
# re-rendered it a second time. This version returns `save_path` (a string)
# instead and always closes the figure after showing it once.

def plot_prediction_bars(pred_result, sample_id='', save_path=None):
    preds  = pred_result['predictions']
    gt     = pred_result['ground_truth']
    x      = range(len(TARGET_NAMES))
    p_vals = [preds[n] for n in TARGET_NAMES]
    g_vals = [gt[n]    for n in TARGET_NAMES]
    fig, ax = plt.subplots(figsize=(10, 4))
    w = 0.35
    b_gt   = ax.bar([i - w/2 for i in x], g_vals, w,
                    label='Thực tế', color=COLOR_SCHEMES['bar_gt'], alpha=0.85)
    b_pred = ax.bar([i + w/2 for i in x], p_vals, w,
                    label='Dự đoán', color=COLOR_SCHEMES['bar_pred'], alpha=0.85)
    for b in list(b_gt) + list(b_pred):
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.1,
                f'{b.get_height():.1f}', ha='center', va='bottom', fontsize=8)
    ax.set_xticks(list(x))
    ax.set_xticklabels(DISPLAY_NAMES, rotation=20, ha='right', fontsize=9)
    ax.set_ylabel('Điểm (1–10)', fontsize=10)
    ax.set_ylim(0, 12)
    ax.set_title(f'{sample_id} — Dự đoán vs Thực tế', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    fig.tight_layout()
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        fig.savefig(save_path, dpi=DEFAULT_DPI, bbox_inches='tight', facecolor='white')
        print(f'[XAI] Đã lưu: {save_path}')
    plt.show()
    plt.close(fig)
    return save_path

# ── Hiển thị ảnh review ──────────────────────────────────────────────────────

def show_review_images(sample, sample_id='', max_show=4):
    n = min(sample['num_real_images'], max_show)
    if n == 0:
        print('  (Không có ảnh thực)')
        return
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
    if n == 1:
        axes = [axes]
    for i in range(n):
        img = sample['loaded_images'][i].convert('RGB').resize((224, 224))
        axes[i].imshow(np.array(img))
        axes[i].set_title(f'Ảnh {i+1}', fontsize=9)
        axes[i].axis('off')
    fig.suptitle(f'{sample_id} — {n} ảnh thực', fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.show()
    plt.close(fig)

# ── Tóm tắt trạng thái artifact cho một mẫu ──────────────────────────────────

def print_sample_summary(sample_id, pred_result, artifact_check):
    print(f'\n{chr(9473)*55}')
    print(f'  TÓM TẮT — {sample_id}')
    print(f'{chr(9473)*55}')
    for key, label in [('gradcam','Grad-CAM'),('attention','Attention'),
                        ('cross_attention','Cross-Attention'),('shap','SHAP'),('lime','LIME')]:
        status = '✅' if artifact_check.get(key) else '⚠️ '
        print(f'  {status} {label}')
    print(f"  Completeness: {artifact_check.get('completeness',0)*100:.0f}%")
    print(f"  MAE: {pred_result['mean_mae']:.3f}")
    for n, d in zip(TARGET_NAMES, DISPLAY_NAMES):
        e = pred_result['absolute_errors'][n]
        mark = '✅' if e <= 0.5 else ('⚠️ ' if e <= 1.5 else '❌')
        print(f'    {mark} {d}: AE={e:.2f}')
    print(f'{chr(9473)*55}\n')

print('✅ Hàm tiện ích đã định nghĩa.')

# ── Shared robust helpers used across every XAI section ──────────────────────

def sample_to_batch(sample):
    '''Adapt load_single_sample output to extract_fused_embeddings input.'''
    labels = sample['factor_scores']
    if labels.dim() == 1:
        labels = labels.unsqueeze(0)
    return {
        'input_ids': sample['input_ids'],
        'attention_mask': sample['attention_mask'],
        'pixel_values': sample['pixel_values'],
        'num_images': sample['num_images'],
        'labels': labels,
    }


def _validate_background_tensor(value):
    if isinstance(value, dict):
        for key in ('background', 'background_fused', 'fused_embeddings'):
            if key in value:
                value = value[key]
                break
    if not isinstance(value, torch.Tensor):
        return None
    value = value.detach().cpu().float()
    if value.dim() != 2 or value.shape[1] != FUSED_DIM or value.shape[0] < 2:
        return None
    if not torch.isfinite(value).all():
        return None
    return value


def prepare_shap_background(candidate_indices, max_samples=8):
    '''Load a valid cached SHAP baseline or build one from multiple samples.'''
    cached_path = os.path.join(XAI_DIR, 'shap', 'raw', 'background_fused.pt')
    if os.path.isfile(cached_path):
        try:
            try:
                cached = torch.load(
                    cached_path, map_location='cpu', weights_only=True
                )
            except TypeError:
                cached = torch.load(cached_path, map_location='cpu')
            cached = _validate_background_tensor(cached)
            if cached is not None:
                print(
                    f'[SHAP] Reusing background: {cached_path} '
                    f'(shape={tuple(cached.shape)})'
                )
                return cached
            print(f'[SHAP] Cached background has an invalid shape: {cached_path}')
        except Exception as exc:
            print(
                f'[SHAP] Cannot load cached background '
                f'({type(exc).__name__}: {exc}); rebuilding.'
            )

    unique_indices = list(dict.fromkeys(int(i) for i in candidate_indices))
    if len(unique_indices) > max_samples:
        positions = np.linspace(
            0, len(unique_indices) - 1, num=max_samples, dtype=int
        )
        unique_indices = [unique_indices[pos] for pos in positions]

    batches = []
    failures = []
    for idx in unique_indices:
        try:
            sample = load_single_sample(
                csv_path=CSV_TEST,
                idx=idx,
                tokenizer=tokenizer,
                image_processor=image_processor,
                image_dir=IMAGE_DIR,
                device=device,
            )
            batches.append(sample_to_batch(sample))
        except Exception as exc:
            failures.append(f'idx={idx}: {type(exc).__name__}: {exc}')

    if len(batches) < 2:
        print(
            '[SHAP] Need at least 2 valid baseline samples; '
            f'found {len(batches)}. SHAP will be skipped safely.'
        )
        for message in failures[:3]:
            print(f'  - {message}')
        return None

    result = run_safe(
        extract_fused_embeddings,
        step_name='prepare_SHAP_background',
        fallback=(None, None, None),
        model=model,
        dataloader=batches,
        device=device,
        max_samples=len(batches),
    )
    fused = result[0] if result else None
    fused = _validate_background_tensor(fused)
    if fused is None:
        print('[SHAP] Background embedding extraction failed; SHAP will be skipped.')
        return None

    background, _ = select_background(
        fused,
        n_background=min(max_samples, fused.shape[0]),
        seed=DEFAULT_SEED,
    )
    print(f'[SHAP] Built multi-sample background: {tuple(background.shape)}')
    return background


def _cross_attention_word_groups(tokens):
    '''Return readable word labels and source-token index groups.'''
    special = {'<s>', '</s>', '<pad>', '<unk>', '<mask>'}
    mode = (
        'at_sign' if any('@@' in token for token in tokens)
        else 'g_prefix' if any(token.startswith('Ġ') for token in tokens)
        else 'none'
    )
    labels, groups = [], []

    if mode == 'at_sign':
        parts, indices = [], []
        for idx, token in enumerate(tokens):
            if token in special:
                continue
            parts.append(token[:-2] if token.endswith('@@') else token)
            indices.append(idx)
            if not token.endswith('@@'):
                labels.append(''.join(parts))
                groups.append(indices)
                parts, indices = [], []
        if indices:
            labels.append(''.join(parts))
            groups.append(indices)
    elif mode == 'g_prefix':
        current_label, indices = '', []
        for idx, token in enumerate(tokens):
            if token in special:
                continue
            if token.startswith('Ġ') and indices:
                labels.append(current_label)
                groups.append(indices)
                current_label, indices = token[1:], [idx]
            else:
                current_label += token[1:] if token.startswith('Ġ') else token
                indices.append(idx)
        if indices:
            labels.append(current_label)
            groups.append(indices)
    else:
        for idx, token in enumerate(tokens):
            if token not in special:
                labels.append(token)
                groups.append([idx])

    clean = [
        (label.strip() or f'word_{position}', group)
        for position, (label, group) in enumerate(zip(labels, groups))
    ]
    return [item[0] for item in clean], [item[1] for item in clean]


def merge_cross_attention_words(tokens, t2i_attn, i2t_attn):
    '''Aggregate subwords for lecturer-facing T→P and P→T displays.'''
    labels, groups = _cross_attention_word_groups(tokens)
    if not groups:
        raise ValueError('No non-special tokens available for cross-attention.')

    word_t2i = np.stack(
        [np.asarray(t2i_attn[group]).mean(axis=0) for group in groups],
        axis=0,
    )
    word_i2t = np.stack(
        [np.asarray(i2t_attn[:, group]).sum(axis=1) for group in groups],
        axis=1,
    )
    row_sums = word_i2t.sum(axis=1, keepdims=True)
    word_i2t = np.divide(
        word_i2t,
        row_sums,
        out=np.zeros_like(word_i2t),
        where=row_sums > 1e-12,
    )
    return labels, word_t2i, word_i2t


def save_lime_text_bar(raw_weights, save_path, title):
    '''Save the standalone text artifact expected by check_sample_artifacts.'''
    if not raw_weights:
        return None
    pairs = sorted(raw_weights, key=lambda item: abs(item[1]))[-10:]
    words = [word for word, _ in pairs]
    values = [float(weight) for _, weight in pairs]
    colors = ['#4CAF50' if value >= 0 else '#F44336' for value in values]
    fig, ax = plt.subplots(figsize=(8, max(4, 0.45 * len(words))))
    ax.barh(range(len(words)), values, color=colors)
    ax.set_yticks(range(len(words)))
    ax.set_yticklabels(words)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlabel('LIME weight (local sensitivity)')
    ax.set_title(title, fontweight='bold')
    fig.tight_layout()
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    fig.savefig(save_path, dpi=DEFAULT_DPI, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    return save_path

---
# 📋 PHẦN 1 — MẪU DEMO CỐ ĐỊNH (sample_0000)

Notebook này dùng đúng một mẫu cố định, không quét hay so sánh nhiều mẫu:

```python
SAMPLE_INDEX = 0
SAMPLE_ID = "sample_0000"
CASE_TYPE = "fixed_test_index_0"
```

Mẫu này được kỳ vọng là review về **bánh canh cua**. Kiểm tra bên dưới chỉ in
cảnh báo (không dừng notebook) nếu nội dung không khớp — để tránh chặn demo vì
lệch phiên bản `data.zip` nhỏ, đồng thời vẫn cảnh báo rõ cho người trình bày.

In [ ]:
SAMPLE_INDEX = 0
SAMPLE_ID = 'sample_0000'
CASE_TYPE = 'fixed_test_index_0'

df_test = pd.read_csv(CSV_TEST)
print(f'Tổng mẫu test: {len(df_test)}')

required_dataset_columns = ['comment_clean', 'image_url', *LABEL_COLS]
missing_dataset_columns = [
    column for column in required_dataset_columns
    if column not in df_test.columns
]
if missing_dataset_columns:
    raise ValueError(
        f'Test CSV thiếu các cột bắt buộc: {missing_dataset_columns}'
    )
if df_test.empty:
    raise ValueError(f'Test CSV không có dòng dữ liệu: {CSV_TEST}')
if not (0 <= SAMPLE_INDEX < len(df_test)):
    raise IndexError(
        f'SAMPLE_INDEX={SAMPLE_INDEX} nằm ngoài phạm vi test set '
        f'(0..{len(df_test) - 1}).'
    )

gradcam_dir   = os.path.join(XAI_DIR, 'gradcam', SAMPLE_ID)
attention_dir = os.path.join(XAI_DIR, 'attention', SAMPLE_ID)
crossattn_dir = os.path.join(XAI_DIR, 'cross_attention', SAMPLE_ID)
shap_dir      = os.path.join(XAI_DIR, 'shap', SAMPLE_ID)
lime_dir      = os.path.join(XAI_DIR, 'lime', SAMPLE_ID)
demo_dir      = os.path.join(DEMO_OUT, SAMPLE_ID)
agent_dir     = os.path.join(AGENT_DIR, SAMPLE_ID)
for output_dir in [
    gradcam_dir, attention_dir, crossattn_dir,
    shap_dir, lime_dir, demo_dir, agent_dir,
]:
    os.makedirs(output_dir, exist_ok=True)

print(f'Đang tải {SAMPLE_ID} (idx={SAMPLE_INDEX}, case_type={CASE_TYPE}) ...')
sample = load_single_sample(
    csv_path=CSV_TEST,
    idx=SAMPLE_INDEX,
    tokenizer=tokenizer,
    image_processor=image_processor,
    image_dir=IMAGE_DIR,
    device=device,
)

# ── One-sample invariants: text non-empty, real images present, labels finite ─
if not str(sample.get('text', '')).strip():
    raise ValueError(f'{SAMPLE_ID}: review text rỗng.')
if sample.get('num_real_images', 0) < 1:
    raise ValueError(f'{SAMPLE_ID}: không có ảnh thực nào (num_real_images=0).')

labels_array = np.asarray(sample['factor_scores'], dtype=float).reshape(-1)
if labels_array.size != len(TARGET_NAMES) or not np.isfinite(labels_array).all():
    raise ValueError(f'{SAMPLE_ID}: nhãn ground-truth không hợp lệ: {labels_array}')

# Non-fatal content check: warn only, never replace the sample or halt.
normalized_review = str(sample['text']).lower()
if 'bánh canh' not in normalized_review or 'cua' not in normalized_review:
    print(
        '[WARN] Test index 0 does not appear to contain the expected '
        'bánh canh cua review. Check that the correct data.zip version '
        'was loaded.'
    )

print(f'  Sample ID    : {SAMPLE_ID}')
print(f'  Test index   : {SAMPLE_INDEX}')
print(f'  Case type    : {CASE_TYPE}')
print(f'  Số ảnh thực  : {sample["num_real_images"]}')
print(
    '  Nhãn thực tế : '
    + ', '.join(f'{n}={v:.1f}' for n, v in zip(TARGET_NAMES, labels_array))
)
print('\n  Nội dung review:')
for line in textwrap.wrap(sample['text'], width=100):
    print(f'    {line}')

show_review_images(sample, SAMPLE_ID)

## 1.1 · Dự đoán

Review và ảnh của `sample_0000` đã hiển thị ở Phần 1 phía trên. Cell dưới đây
chỉ tạo **một** bảng dự đoán và **đúng một** biểu đồ cột — không có biểu đồ
trùng lặp (xem ghi chú sửa lỗi trong `plot_prediction_bars` ở mục 0.9).

In [ ]:
pred_result = get_prediction(model, sample)
display_prediction_table(pred_result, SAMPLE_ID)
_ = plot_prediction_bars(
    pred_result,
    SAMPLE_ID,
    save_path=os.path.join(demo_dir, f'{SAMPLE_ID}_prediction.png'),
)

## 1.2 · Grad-CAM — Vùng ảnh quan trọng

> **Giới hạn:** image encoder dùng chung cho cả 5 đầu ra nên heatmap giữa các
> target có thể rất giống nhau. Demo chỉ hiển thị **Overall Satisfaction**,
> không dùng Grad-CAM để tuyên bố khác biệt per-target hay quan hệ nhân quả.

In [ ]:
# ── Grad-CAM: Only Overall Satisfaction (target_idx=4) ───────────────────────
# Shared encoder → cosine sim >0.95 across all 5 targets → show only overall
import datetime as _dt

TARGET_IDX_GRADCAM = 4
gradcam_results = {}
for img_idx in range(min(sample['num_real_images'], 4)):
    cam = run_safe(
        compute_gradcam_for_image, step_name=f'GradCAM img{img_idx}',
        fallback=None,
        model=model, sample=sample, target_idx=TARGET_IDX_GRADCAM,
        image_idx=img_idx, target_layer=target_layer, device=device,
    )
    gradcam_results[img_idx] = cam

n_show = min(sample['num_real_images'], 2)
if n_show > 0:
    import matplotlib.cm as _cm
    from PIL import Image as _PILI
    fig, axes = plt.subplots(n_show, 3, figsize=(12, 4 * n_show), squeeze=False)
    for img_idx in range(n_show):
        pil_img = sample['loaded_images'][img_idx]
        img224  = np.array(pil_img.convert('RGB').resize((224, 224)))
        cam     = gradcam_results.get(img_idx)

        axes[img_idx][0].imshow(img224)
        axes[img_idx][0].set_title(f'Ảnh {img_idx+1} — Gốc', fontsize=9)
        axes[img_idx][0].axis('off')

        if cam is not None:
            heatmap = _cm.jet(cam)[:, :, :3]
            axes[img_idx][1].imshow(heatmap)
            axes[img_idx][1].set_title('Grad-CAM Heatmap', fontsize=9)
        else:
            axes[img_idx][1].text(0.5, 0.5, 'N/A', ha='center', va='center',
                                   transform=axes[img_idx][1].transAxes)
        axes[img_idx][1].axis('off')

        if cam is not None:
            overlay = run_safe(overlay_cam_on_image, step_name='overlay',
                               cam=cam, original_image=pil_img,
                               image_size=224, colormap_name='jet', alpha=0.5)
            if overlay is not None:
                axes[img_idx][2].imshow(overlay)
                axes[img_idx][2].set_title('Overlay (CAM + Ảnh)', fontsize=9)
                cam_save = f'{gradcam_dir}/gradcam_img{img_idx}_overall.png'
                _PILI.fromarray(overlay).save(cam_save)
            else:
                axes[img_idx][2].axis('off')
        else:
            axes[img_idx][2].axis('off')

    fig.suptitle(f'Grad-CAM — {SAMPLE_ID} (Overall Satisfaction)',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    gcpath = f'{demo_dir}/{SAMPLE_ID}_gradcam_3panel.png'
    fig.savefig(gcpath, dpi=DEFAULT_DPI, bbox_inches='tight', facecolor='white')
    plt.show()
    plt.close(fig)
    print(f'[XAI] Đã lưu: {gcpath}')
else:
    print('[SKIP] Không có ảnh → bỏ qua Grad-CAM')

# metadata.json cho EvidenceLoader (chỉ ghi nếu có ít nhất 1 ảnh được tính)
if any(v is not None for v in gradcam_results.values()):
    gradcam_meta = {
        'sample_id': SAMPLE_ID,
        'sample_idx': SAMPLE_INDEX,
        'num_images': sample['num_real_images'],
        'num_targets': 1,
        'target_names': ['overall'],
        'display_names': ['Overall Satisfaction'],
        'target_layer': type(target_layer).__name__,
        'device': str(device),
        'timestamp': _dt.datetime.now().isoformat(),
        'artifacts': {
            f'img{i}_overall': f'gradcam_img{i}_overall.png'
            for i, v in gradcam_results.items() if v is not None
        },
    }
    with open(f'{gradcam_dir}/metadata.json', 'w', encoding='utf-8') as f:
        json.dump(gradcam_meta, f, ensure_ascii=False, indent=2)

## 1.3 · PhoBERT Attention — Mức từ đã gộp

> Output hiển thị đã gộp BPE/subword thành từ đọc được (không hiển thị mảnh
> `ng@@`, `nư@@`...). Attention mô tả luồng thông tin, không tự nó chứng minh
> quan hệ nhân quả.

In [ ]:
# ── PhoBERT Attention: CLS → merged word-level importance ────────────────────
attn_result = run_safe(
    extract_phobert_attention,
    step_name='extract_phobert_attention',
    fallback=None,
    model=model,
    input_ids=sample['input_ids'],
    attention_mask=sample['attention_mask'],
    tokenizer=tokenizer,
)

word_importances = []
if attn_result is not None:
    attentions = attn_result['attentions']
    tokens = attn_result['tokens']
    seq_len = attn_result['seq_len']
    print(
        f'  tokens={seq_len}, '
        f'attention shape={attentions.shape}'
    )

    agg_matrix = aggregate_attention(
        attentions, strategy='last_layer_mean'
    )
    cls_result = cls_token_importance(
        agg_matrix, tokens
    )
    word_importances = merge_subword_attention(
        cls_result['importances'],
        tokens,
        strategy='mean',
    )

    print(f'Top 10 từ đã gộp subword ({SAMPLE_ID}):')
    for word, score in word_importances[:10]:
        print(f'  {word:<24s} {score:.4f}')

    word_labels = [word for word, _ in word_importances]
    word_values = [value for _, value in word_importances]
    bar_path = os.path.join(
        attention_dir, 'cls_importance_word_bar.png'
    )
    bar_fig = run_safe(
        plot_cls_importance_bar,
        step_name='plot_word_attention',
        fallback=None,
        tokens=word_labels,
        importances=word_values,
        title=f'PhoBERT Word-Level Attention — {SAMPLE_ID}',
        save_path=None,
        top_k=15,
        dpi=DEFAULT_DPI,
    )
    if bar_fig is not None:
        bar_fig.savefig(
            bar_path,
            dpi=DEFAULT_DPI,
            bbox_inches='tight',
            facecolor='white',
        )
        plt.show()
        plt.close(bar_fig)
        print(f'[XAI] Đã lưu: {bar_path}')

    # Raw tensors are retained for audit/reuse, but are not shown to lecturers.
    np.savez_compressed(
        os.path.join(attention_dir, 'raw_attention.npz'),
        attentions=attentions,
        tokens=np.asarray(tokens, dtype=object),
    )
    word_payload = {
        'word_importances': [
            {'word': word, 'importance': float(score)}
            for word, score in word_importances
        ]
    }
    with open(
        os.path.join(attention_dir, 'word_importance.json'),
        'w',
        encoding='utf-8',
    ) as file:
        json.dump(word_payload, file, ensure_ascii=False, indent=2)
    with open(
        os.path.join(attention_dir, 'topk_tokens.json'),
        'w',
        encoding='utf-8',
    ) as file:
        json.dump(
            [
                {'token': word, 'importance': float(score)}
                for word, score in word_importances[:15]
            ],
            file,
            ensure_ascii=False,
            indent=2,
        )
    print(
        '[Attention] Visible output uses merged words; '
        'raw BPE/subword fragments are stored only as machine-readable data.'
    )
else:
    print('[SKIP] Không trích xuất được PhoBERT attention.')

## 1.4 · Cross-Attention (phiên bản cải tiến) — Tương tác hai chiều

- **Token → Patch:** khi xử lý một từ, mô hình phân bổ chú ý lên các patch ảnh nào?
- **Patch → Token:** từ một patch ảnh, mô hình liên kết ngược tới các từ nào?

Hai hướng dùng phép chuẩn hóa khác nhau (softmax theo hai trục khác nhau);
`i2t_attn` **không** phải là chuyển vị của `t2i_attn`.

> **Kiến trúc quan trọng cần biết:** `ImageModel.forward_features` (xem
> `models/ImageModel.py`) **gộp trung bình (mean-pool)** các patch feature
> của toàn bộ ảnh thực **trước khi** đưa vào cross-attention. Do đó cross-attention
> chỉ nhìn thấy **một** lưới patch duy nhất (không nhân với số ảnh); một patch
> index tương ứng với **cùng một vị trí không gian trên mọi ảnh thực** của mẫu,
> chứ không thuộc riêng một ảnh cụ thể. Hai hình bên dưới phản ánh trung thực
> điều này thay vì gán patch cho một ảnh cụ thể một cách sai lệch.

Notebook thay thế heatmap ma trận thô (khó đọc trước lớp) bằng **hai hình
trực quan cho giảng viên**:
1. **Text → Image** — các từ quan trọng nhất và vùng ảnh chúng chú ý tới.
2. **Image → Text** — các patch ảnh nổi bật nhất và các từ liên kết với chúng.

Ma trận attention thô vẫn được lưu đầy đủ để tái lập kết quả.

In [ ]:
# ── Improved Cross-Attention: extraction, word merging, semantic filtering ───
import re as _re
from PIL import Image as _PILImgCA

cross_result = run_safe(
    extract_cross_attention,
    step_name='extract_cross_attention',
    fallback=None,
    model=model,
    sample=sample,
    tokenizer=tokenizer,
)

# Small explicit Vietnamese stopword filter used ONLY for lecturer-facing
# display ranking. It never modifies the underlying raw attention values.
VIETNAMESE_STOPWORDS = {
    'là', 'và', 'của', 'có', 'được', 'cho', 'thì', 'mà', 'nên', 'này',
    'đó', 'các', 'một', 'những', 'rất', 'khi', 'nhưng', 'nếu', 'với',
    'ở', 'ra', 'vào', 'lên', 'xuống', 'the', 'a', 'an', 'to', 'of',
}
_WORD_CHAR_RE = _re.compile(
    r'[a-zA-Zàáạảãâầấậẩẫăằắặẳẵèéẹẻẽêềếệểễìíịỉĩòóọỏõôồốộổỗơờớợởỡ'
    r'ùúụủũưừứựửữỳýỵỷỹđ0-9]'
)


def is_display_word(word):
    '''Filter used only for lecturer-facing ranking, not for raw data.'''
    w = word.strip().lower()
    if not w or not _WORD_CHAR_RE.search(w):
        return False  # empty / punctuation-only / malformed fragment
    return w not in VIETNAMESE_STOPWORDS


t2i = i2t = raw_tokens = ca_words = word_t2i = word_i2t = None
num_patches = grid_size = 0
patch_grid_valid = False

if cross_result is not None:
    t2i = cross_result['t2i_attn']
    i2t = cross_result['i2t_attn']
    raw_tokens = cross_result['tokens']

    word_merge = run_safe(
        merge_cross_attention_words,
        step_name='merge_cross_attention_words',
        fallback=([], None, None),
        tokens=raw_tokens,
        t2i_attn=t2i,
        i2t_attn=i2t,
    )
    ca_words, word_t2i, word_i2t = word_merge

    if word_t2i is not None and word_i2t is not None:
        num_patches = word_t2i.shape[1]
        grid_size = int(round(np.sqrt(num_patches)))
        patch_grid_valid = (grid_size * grid_size == num_patches)

        # Raw (unfiltered, token-level) attention is always preserved as-is.
        np.savez_compressed(
            os.path.join(crossattn_dir, 'cross_attention_raw.npz'),
            t2i=t2i,
            i2t=i2t,
            word_t2i=word_t2i,
            word_i2t=word_i2t,
            words=np.asarray(ca_words, dtype=object),
        )

        grid_desc = (
            f'{grid_size}x{grid_size}' if patch_grid_valid
            else 'non-square — spatial visualization will be skipped'
        )
        print(
            f'[Cross-Attention] {len(raw_tokens)} raw tokens -> '
            f'{len(ca_words)} merged words; P={num_patches} patches ({grid_desc}).'
        )
        print(
            '[Cross-Attention] NOTE: image patch features are mean-pooled across '
            f'all {sample["num_real_images"]} real image(s) of this sample before '
            'cross-attention (models/ImageModel.py). A patch index therefore maps '
            'to the same spatial location in every real image, not to one '
            'exclusive source image.'
        )
    else:
        print('[SKIP] Không thể gộp Cross-Attention ở mức từ.')
else:
    print('[SKIP] Không trích xuất được Cross-Attention.')

In [ ]:
# ── Figure 1: Text → Image — top semantic words and attended image regions ───
top_tokens_overlay_path = os.path.join(
    crossattn_dir, 'top_tokens_patch_overlay_grid.png'
)
token_patch_topk_path = os.path.join(crossattn_dir, 'token_patch_topk.json')
NUM_TOP_WORDS = 5

selected_word_records = []

if patch_grid_valid and word_t2i is not None:
    # Ranking method: max Token→Patch attention across all patches per word.
    word_strength = word_t2i.max(axis=1)
    candidate_order = np.argsort(word_strength)[::-1]

    occurrence_total = {}
    selected_indices = []
    for wi in candidate_order:
        label = ca_words[wi]
        if not is_display_word(label):
            continue
        selected_indices.append(wi)
        key = label.strip().lower()
        occurrence_total[key] = occurrence_total.get(key, 0) + 1
        if len(selected_indices) >= NUM_TOP_WORDS:
            break

    if not selected_indices:
        print('[Cross-Attention] Không có từ nào vượt qua bộ lọc hiển thị; bỏ qua Figure 1.')
    else:
        _SUBSCRIPTS = '₀₁₂₃₄₅₆₇₈₉'

        def _subscript(n):
            return ''.join(_SUBSCRIPTS[int(d)] for d in str(n))

        running_count = {}
        display_labels = []
        for wi in selected_indices:
            key = ca_words[wi].strip().lower()
            running_count[key] = running_count.get(key, 0) + 1
            label = ca_words[wi]
            # Disambiguate repeated words (e.g. "dai" appearing twice) instead
            # of silently merging distinct occurrences into one label.
            if occurrence_total[key] > 1:
                label = f'{label}{_subscript(running_count[key])}'
            display_labels.append(label)

        base_image = sample['loaded_images'][0].convert('RGB').resize((224, 224))
        base_np = np.array(base_image, dtype=np.float32) / 255.0

        heatmaps, peak_scores, peak_patches = [], [], []
        for wi in selected_indices:
            row = word_t2i[wi]
            heat = row.reshape(grid_size, grid_size).astype(np.float32)
            heat_img = _PILImgCA.fromarray(heat, mode='F')
            heat_resized = np.array(
                heat_img.resize((224, 224), _PILImgCA.BILINEAR), dtype=np.float32
            )
            heatmaps.append(heat_resized)
            peak_idx = int(np.argmax(row))
            peak_scores.append(float(row[peak_idx]))
            peak_patches.append(peak_idx)

        # Shared normalization range across all panels (requirement: one colorbar).
        vmax = max(peak_scores) if peak_scores else 1.0

        n_panels = len(selected_indices)
        fig, axes = plt.subplots(1, n_panels, figsize=(3.6 * n_panels, 4.2))
        if n_panels == 1:
            axes = [axes]

        cmap = plt.cm.inferno
        ph = pw = 224 // grid_size
        for ax, heat, label, score, patch_idx in zip(
            axes, heatmaps, display_labels, peak_scores, peak_patches
        ):
            ax.imshow(base_np)
            ax.imshow(heat, cmap=cmap, alpha=0.55, vmin=0, vmax=vmax)
            row_p, col_p = divmod(patch_idx, grid_size)
            rect = plt.Rectangle(
                (col_p * pw, row_p * ph), pw, ph,
                linewidth=2, edgecolor='cyan', facecolor='none',
            )
            ax.add_patch(rect)
            img_note = (
                'Image 1' if sample['num_real_images'] == 1
                else f'Image 1 (pooled across {sample["num_real_images"]} images)'
            )
            ax.set_title(f'"{label}"\n{img_note}\npeak attention: {score:.3f}', fontsize=9)
            ax.axis('off')

        sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=0, vmax=vmax))
        sm.set_array([])
        fig.colorbar(sm, ax=axes, shrink=0.7, label='Token→Patch attention')

        fig.suptitle(
            'Text → Image Cross-Attention\n'
            'Important review words and their attended image regions',
            fontsize=13, fontweight='bold',
        )
        os.makedirs(os.path.dirname(top_tokens_overlay_path), exist_ok=True)
        fig.savefig(
            top_tokens_overlay_path, dpi=THESIS_DPI,
            bbox_inches='tight', facecolor='white',
        )
        plt.show()
        plt.close(fig)
        print(f'[XAI] Đã lưu: {top_tokens_overlay_path}')

        for wi, label, score, patch_idx in zip(
            selected_indices, display_labels, peak_scores, peak_patches
        ):
            row_p, col_p = divmod(patch_idx, grid_size)
            selected_word_records.append({
                'word': label,
                'raw_word': ca_words[wi],
                'word_index': int(wi),
                'patch_index': int(patch_idx),
                'patch_row': int(row_p),
                'patch_col': int(col_p),
                'attention': float(score),
            })

        with open(token_patch_topk_path, 'w', encoding='utf-8') as f:
            json.dump(selected_word_records, f, ensure_ascii=False, indent=2)
        print(f'[XAI] Đã lưu: {token_patch_topk_path}')
else:
    print('[SKIP] Patch grid không hợp lệ hoặc thiếu dữ liệu → bỏ qua Figure 1 (Text→Image).')

In [ ]:
# ── Figure 2: Image → Text — top image patches and associated review words ───
top_patches_ranking_path = os.path.join(
    crossattn_dir, 'top_patches_token_rankings.png'
)
patch_token_topk_path = os.path.join(crossattn_dir, 'patch_token_topk.json')
NUM_TOP_PATCHES = 3
NUM_WORDS_PER_PATCH = 5

selected_patch_records = []

if patch_grid_valid and word_i2t is not None and i2t is not None:
    # Ranking method: total raw Image→Text attention issued by each patch.
    patch_salience = i2t.sum(axis=1)
    top_patch_indices = np.argsort(patch_salience)[::-1][:NUM_TOP_PATCHES]

    base_image = sample['loaded_images'][0].convert('RGB').resize((224, 224))
    base_np = np.array(base_image, dtype=np.float32) / 255.0
    ph = pw = 224 // grid_size

    fig, axes = plt.subplots(
        len(top_patch_indices), 3, figsize=(13, 4.2 * len(top_patch_indices)),
        squeeze=False, gridspec_kw={'width_ratios': [1.1, 0.9, 1.6]},
    )

    for row_i, patch_idx in enumerate(top_patch_indices):
        patch_idx = int(patch_idx)
        pr, pc = divmod(patch_idx, grid_size)
        y0, x0 = pr * ph, pc * pw

        ax_img = axes[row_i][0]
        ax_img.imshow(base_np)
        rect = plt.Rectangle((x0, y0), pw, ph, linewidth=2.5,
                              edgecolor='lime', facecolor=(0, 1, 0, 0.2))
        ax_img.add_patch(rect)
        img_note = (
            'Image 1' if sample['num_real_images'] == 1
            else f'Image 1 (pooled across {sample["num_real_images"]} images)'
        )
        ax_img.set_title(f'Patch {patch_idx} (row={pr}, col={pc})\n{img_note}', fontsize=9)
        ax_img.axis('off')

        ax_zoom = axes[row_i][1]
        crop = base_np[y0:y0 + ph, x0:x0 + pw]
        ax_zoom.imshow(crop)
        ax_zoom.set_title('Zoomed patch', fontsize=9)
        ax_zoom.axis('off')

        word_scores = word_i2t[patch_idx]
        order = np.argsort(word_scores)[::-1]
        picked = []
        for wi in order:
            label = ca_words[wi]
            if not is_display_word(label):
                continue
            picked.append((int(wi), label, float(word_scores[wi])))
            if len(picked) >= NUM_WORDS_PER_PATCH:
                break

        ax_bar = axes[row_i][2]
        if picked:
            bar_labels = [label for _, label, _ in reversed(picked)]
            bar_values = [score for _, _, score in reversed(picked)]
            ax_bar.barh(
                range(len(bar_labels)), bar_values,
                color=COLOR_SCHEMES['modality_colors']['image'], edgecolor='white',
            )
            ax_bar.set_yticks(range(len(bar_labels)))
            ax_bar.set_yticklabels(bar_labels, fontsize=9)
            ax_bar.set_xlabel('Image→Text attention (normalized share)', fontsize=8)
            ax_bar.set_xlim(0, max(bar_values) * 1.25 if bar_values else 1.0)
            for y_pos, val in enumerate(bar_values):
                ax_bar.text(val, y_pos, f' {val:.3f}', va='center', fontsize=7)
        else:
            ax_bar.text(0.5, 0.5, 'N/A', ha='center', va='center', transform=ax_bar.transAxes)
            ax_bar.axis('off')
        ax_bar.set_title('Top associated words', fontsize=9, fontweight='bold')

        selected_patch_records.append({
            'patch_index': patch_idx,
            'patch_row': int(pr),
            'patch_col': int(pc),
            'salience': float(patch_salience[patch_idx]),
            'top_words': [
                {'word': label, 'raw_word': ca_words[wi], 'attention': score}
                for wi, label, score in picked
            ],
        })

    fig.suptitle(
        'Image → Text Cross-Attention\n'
        'Important visual patches and their associated review words',
        fontsize=13, fontweight='bold',
    )
    fig.tight_layout(rect=[0, 0, 1, 0.94])
    os.makedirs(os.path.dirname(top_patches_ranking_path), exist_ok=True)
    fig.savefig(
        top_patches_ranking_path, dpi=THESIS_DPI,
        bbox_inches='tight', facecolor='white',
    )
    plt.show()
    plt.close(fig)
    print(f'[XAI] Đã lưu: {top_patches_ranking_path}')

    with open(patch_token_topk_path, 'w', encoding='utf-8') as f:
        json.dump(selected_patch_records, f, ensure_ascii=False, indent=2)
    print(f'[XAI] Đã lưu: {patch_token_topk_path}')
else:
    print('[SKIP] Patch grid không hợp lệ hoặc thiếu dữ liệu → bỏ qua Figure 2 (Image→Text).')

In [ ]:
# ── Cross-Attention interpretation & summary artifacts ───────────────────────
cross_attention_summary_path = os.path.join(
    crossattn_dir, 'cross_attention_summary.json'
)

CROSS_ATTENTION_LIMITATION = (
    'Cross-attention visualizes internal associations used to construct the '
    'fused representation. It is not definitive causal attribution and is not '
    'target-specific unless an additional target-conditioning method is '
    'explicitly applied.'
)

if patch_grid_valid and selected_word_records and selected_patch_records:
    print('Cross-modal observations:')
    for rec in selected_word_records:
        print(
            f'- The word "{rec["word"]}" attends most strongly to patch '
            f'(row={rec["patch_row"]}, col={rec["patch_col"]}), '
            f'peak attention={rec["attention"]:.3f}.'
        )
    for rec in selected_patch_records:
        top_words_str = ', '.join(f'"{w["word"]}"' for w in rec['top_words'][:3])
        if top_words_str:
            print(
                f'- Patch (row={rec["patch_row"]}, col={rec["patch_col"]}) is most '
                f'associated with {top_words_str}.'
            )
    print(
        'Both directions show an alignment between food-related review words '
        'and the visible dish region(s) of the sample images.'
    )
    print(f'\n[LIMITATION] {CROSS_ATTENTION_LIMITATION}')
else:
    print('[SKIP] Không đủ dữ liệu để in diễn giải Cross-Attention.')

cross_attention_summary = {
    'sample_id': SAMPLE_ID,
    'sample_index': SAMPLE_INDEX,
    'num_real_images': int(sample['num_real_images']),
    'num_raw_tokens': int(len(raw_tokens)) if raw_tokens is not None else 0,
    'num_merged_words': int(len(ca_words)) if ca_words is not None else 0,
    'total_patches': int(num_patches),
    'patches_per_image': int(num_patches),
    'patch_grid_shape': [grid_size, grid_size] if patch_grid_valid else None,
    'spatial_mapping_validated': bool(patch_grid_valid),
    'patch_pooling_note': (
        'ImageModel.forward_features mean-pools patch features across all real '
        'images of a sample before cross-attention; a patch index corresponds '
        'to the same spatial location in every real image, not to one '
        'exclusive source image.'
    ),
    'display_filtering_rules': {
        'exclude_special_tokens': ['<s>', '</s>', '<pad>', '<unk>', '<mask>'],
        'exclude_punctuation_or_non_alphanumeric': True,
        'stopword_filter': sorted(VIETNAMESE_STOPWORDS),
    },
    'ranking_method': {
        'top_words': 'max Token→Patch attention across all patches',
        'top_patches': 'sum of raw Image→Text attention across all real tokens',
    },
    'selected_top_words': selected_word_records,
    'selected_top_patches': selected_patch_records,
    'scientific_limitation': CROSS_ATTENTION_LIMITATION,
}
with open(cross_attention_summary_path, 'w', encoding='utf-8') as f:
    json.dump(cross_attention_summary, f, ensure_ascii=False, indent=2)
print(f'[XAI] Đã lưu: {cross_attention_summary_path}')

## 1.5 · SHAP Background — baseline đa mẫu

SHAP cần một phân phối baseline nhiều mẫu để không tạo attribution bằng 0.
Notebook **không** quét 200 mẫu chỉ để chọn mẫu demo — nó chỉ tải tối đa 8 mẫu
hợp lệ trong khoảng idx 1–32 (loại trừ `SAMPLE_INDEX=0`) làm baseline, dùng
đúng `prepare_shap_background` đã định nghĩa ở mục 0.9.

In [ ]:
max_candidate_idx = min(32, len(df_test) - 1)
shap_candidate_indices = [
    i for i in range(1, max_candidate_idx + 1) if i != SAMPLE_INDEX
]

if not shap_candidate_indices:
    print('[SHAP] Không đủ mẫu test khác để xây baseline; SHAP sẽ skip an toàn.')
    SHAP_BACKGROUND = None
else:
    SHAP_BACKGROUND = prepare_shap_background(
        shap_candidate_indices, max_samples=8,
    )
    if SHAP_BACKGROUND is None:
        print('[SHAP] Không có baseline hợp lệ; SHAP sẽ skip an toàn.')

## 1.6 · SHAP — Đóng góp fused embedding [1024]

> `text-origin` (dims 0:512) và `image-origin` (dims 512:1024) đều là biểu diễn
> **sau Cross-Attention**. Đây không phải hai modality thuần túy. SHAP dùng
> baseline nhiều mẫu ở trên, không dùng chính sample làm baseline.

In [ ]:
# ── SHAP on fused embedding [1024], using a multi-sample baseline ─────────────
fused_result = run_safe(
    extract_fused_embeddings,
    step_name='extract_fused_embeddings',
    fallback=(None, None, None),
    model=model,
    dataloader=[sample_to_batch(sample)],
    device=device,
    max_samples=1,
)
fused = fused_result[0] if fused_result else None

shap_values_by_factor = {}
shap_contrib_by_factor = {}
shap_vals = None
shap_contrib = None

if fused is None:
    print('[SKIP] Không extract được fused embedding.')
elif SHAP_BACKGROUND is None:
    print(
        '[SKIP] SHAP không có multi-sample background hợp lệ; '
        'không dùng chính sample làm baseline vì sẽ tạo attribution bằng 0.'
    )
else:
    print(
        f'[SHAP] Background={tuple(SHAP_BACKGROUND.shape)}, '
        f'sample={tuple(fused.shape)}'
    )
    for score_index, factor_name in enumerate(FACTOR_NAMES):
        wrapper = FusionHeadWrapper(model.head, score_index=score_index)
        shap_result = run_safe(
            compute_shap_values,
            step_name=f'SHAP_{factor_name}',
            fallback=(None, None),
            wrapper=wrapper,
            background=SHAP_BACKGROUND,
            samples=fused,
        )
        values = shap_result[0] if shap_result else None
        if values is None or len(values) == 0:
            continue
        sample_values = np.asarray(values[0], dtype=float).reshape(-1)
        if sample_values.size != FUSED_DIM or not np.isfinite(
            sample_values
        ).all():
            print(
                f'[SKIP] SHAP {factor_name}: invalid shape/value '
                f'{sample_values.shape}'
            )
            continue
        contribution = modality_contribution(sample_values)
        shap_values_by_factor[factor_name] = sample_values
        shap_contrib_by_factor[factor_name] = contribution
        print(
            f'  {DISPLAY_NAMES[score_index]:<22s} '
            f'text-origin={contribution["text_pct"]:5.1f}% | '
            f'image-origin={contribution["image_pct"]:5.1f}%'
        )

    shap_vals = shap_values_by_factor.get('overall')
    shap_contrib = shap_contrib_by_factor.get('overall')

    if shap_contrib_by_factor:
        contribution_path = os.path.join(
            shap_dir, 'shap_modality_contribution.json'
        )
        with open(
            contribution_path, 'w', encoding='utf-8'
        ) as file:
            json.dump(
                shap_contrib_by_factor,
                file,
                ensure_ascii=False,
                indent=2,
            )
        np.savez_compressed(
            os.path.join(shap_dir, 'raw_shap_values.npz'),
            **shap_values_by_factor,
        )
        print(f'[XAI] Đã lưu: {contribution_path}')

        fig, axes = plt.subplots(1, 3, figsize=(18, 5))

        if (
            shap_contrib is not None
            and shap_contrib['text_abs']
            + shap_contrib['image_abs'] > 1e-12
        ):
            axes[0].pie(
                [
                    shap_contrib['text_abs'],
                    shap_contrib['image_abs'],
                ],
                labels=[
                    f'Text-origin\n{shap_contrib["text_pct"]:.1f}%',
                    f'Image-origin\n{shap_contrib["image_pct"]:.1f}%',
                ],
                colors=[
                    COLOR_SCHEMES['modality_colors']['text'],
                    COLOR_SCHEMES['modality_colors']['image'],
                ],
                autopct='%1.1f%%',
                startangle=90,
            )
            axes[0].set_title(
                'Overall Satisfaction\nModality contribution',
                fontweight='bold',
            )
        else:
            axes[0].text(
                0.5, 0.5, 'Overall SHAP unavailable',
                ha='center', va='center', transform=axes[0].transAxes
            )
            axes[0].axis('off')

        available_factors = [
            factor for factor in FACTOR_NAMES
            if factor in shap_contrib_by_factor
        ]
        x_positions = np.arange(len(available_factors))
        text_pct = [
            shap_contrib_by_factor[factor]['text_pct']
            for factor in available_factors
        ]
        image_pct = [
            shap_contrib_by_factor[factor]['image_pct']
            for factor in available_factors
        ]
        axes[1].bar(
            x_positions,
            text_pct,
            label='Text-origin',
            color=COLOR_SCHEMES['modality_colors']['text'],
        )
        axes[1].bar(
            x_positions,
            image_pct,
            bottom=text_pct,
            label='Image-origin',
            color=COLOR_SCHEMES['modality_colors']['image'],
        )
        axes[1].set_xticks(x_positions)
        axes[1].set_xticklabels(
            [
                DISPLAY_NAMES[FACTOR_NAMES.index(factor)]
                for factor in available_factors
            ],
            rotation=25,
            ha='right',
            fontsize=8,
        )
        axes[1].set_ylim(0, 100)
        axes[1].set_ylabel('|SHAP| contribution (%)')
        axes[1].set_title(
            'Per-target origin contribution', fontweight='bold'
        )
        axes[1].legend(fontsize=8)

        if shap_vals is not None:
            top_count = min(20, shap_vals.size)
            top_indices = np.argsort(np.abs(shap_vals))[
                -top_count:
            ][::-1]
            top_values = shap_vals[top_indices]
            dim_labels = [
                (
                    f'T{index}'
                    if index < CROSS_ATTN_HIDDEN_DIM
                    else f'I{index - CROSS_ATTN_HIDDEN_DIM}'
                )
                for index in top_indices
            ]
            colors = [
                (
                    COLOR_SCHEMES['shap_positive']
                    if value >= 0
                    else COLOR_SCHEMES['shap_negative']
                )
                for value in top_values
            ]
            axes[2].barh(
                range(top_count),
                top_values[::-1],
                color=colors[::-1],
            )
            axes[2].set_yticks(range(top_count))
            axes[2].set_yticklabels(dim_labels[::-1], fontsize=7)
            axes[2].axvline(0, color='black', linewidth=0.8)
            axes[2].set_xlabel('SHAP value')
            axes[2].set_title(
                'Overall: top fused dimensions', fontweight='bold'
            )
        else:
            axes[2].text(
                0.5, 0.5, 'Overall SHAP unavailable',
                ha='center', va='center', transform=axes[2].transAxes
            )
            axes[2].axis('off')

        fig.suptitle(
            f'SHAP — {SAMPLE_ID}\n'
            'Origins are cross-attended representations, not pure modalities',
            fontsize=12,
            fontweight='bold',
        )
        fig.tight_layout()
        shap_path = os.path.join(
            demo_dir, f'{SAMPLE_ID}_shap_analysis.png'
        )
        fig.savefig(
            shap_path,
            dpi=DEFAULT_DPI,
            bbox_inches='tight',
            facecolor='white',
        )
        plt.show()
        plt.close(fig)
        print(f'[XAI] Đã lưu: {shap_path}')
    else:
        print('[SKIP] Không target nào tạo được SHAP values hợp lệ.')

## 1.7 · LIME — Giải thích cục bộ (Text + Image)

> LIME đo độ nhạy quanh **mẫu hiện tại** bằng perturbation ngẫu nhiên. Kết quả
> không đại diện cho hành vi toàn cục của mô hình.

In [ ]:
# ── LIME: Text + Image (target=Overall Satisfaction) ─────────────────────────
TARGET_IDX_LIME = 4
lime_text_weights = {}
lime_text_exp     = None
lime_image_exp    = None
lime_img_paths    = {}

# LIME Text
print('[LIME Text] Đang tính ...')
lime_text_exp = run_safe(
    run_lime_text, step_name='LIME_text',
    fallback=None,
    model=model, sample=sample, score_index=TARGET_IDX_LIME,
    tokenizer=tokenizer, device=device,
    num_features=10, num_samples=300,
)
raw_weights = []
if lime_text_exp is not None:
    raw_weights = lime_text_exp.as_list(label=1)
    lime_text_weights = dict(raw_weights)
    print('  Top LIME words:')
    for word, w in sorted(raw_weights, key=lambda x: abs(x[1]), reverse=True)[:8]:
        print(f'    {("+" if w>0 else "-")} {word:<18s} {abs(w):.4f}')
    lime_factor_name = FACTOR_NAMES[TARGET_IDX_LIME]
    lt_path = f'{lime_dir}/{SAMPLE_ID}_lime_text_{lime_factor_name}_weights.json'
    with open(lt_path, 'w', encoding='utf-8') as f:
        json.dump(raw_weights, f, ensure_ascii=False, indent=2)
    print(f'[XAI] Đã lưu: {lt_path}')
    lime_text_bar_path = os.path.join(
        lime_dir,
        f'{SAMPLE_ID}_lime_text_{lime_factor_name}_bar.png',
    )
    save_lime_text_bar(
        raw_weights,
        lime_text_bar_path,
        f'LIME Local Text Explanation — {SAMPLE_ID}',
    )
    print(f'[XAI] Đã lưu: {lime_text_bar_path}')
else:
    print('[SKIP] LIME text thất bại.')

# LIME Image
if sample['num_real_images'] > 0:
    print('[LIME Image] Đang tính ...')
    lime_image_exp = run_safe(
        run_lime_image, step_name='LIME_image',
        fallback=None,
        model=model, sample=sample, score_index=TARGET_IDX_LIME,
        image_processor=image_processor, device=device, num_samples=300,
    )
    if lime_image_exp is not None:
        lime_img_paths = run_safe(
            save_lime_image_explanation, step_name='save_LIME_image',
            fallback={},
            explanation=lime_image_exp,
            original_image=sample['loaded_images'][0],
            save_dir=lime_dir,
            sample_id=SAMPLE_ID,
            target_idx=TARGET_IDX_LIME,
            factor_name=FACTOR_NAMES[TARGET_IDX_LIME],
            dpi=DEFAULT_DPI,
        ) or {}
    else:
        print('[SKIP] LIME image thất bại.')
else:
    print('[SKIP] Không có ảnh → bỏ qua LIME image.')

# 4-panel LIME visualisation
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

# Panel 1: text importance bar
if lime_text_weights:
    w_sorted = sorted(lime_text_weights.items(), key=lambda x: abs(x[1]), reverse=True)[:10]
    w_names  = [w[0] for w in reversed(w_sorted)]
    w_values = [w[1] for w in reversed(w_sorted)]
    bar_cols = [COLOR_SCHEMES['shap_positive'] if v >= 0
                else COLOR_SCHEMES['shap_negative'] for v in w_values]
    axes[0].barh(range(len(w_names)), w_values, color=bar_cols)
    axes[0].set_yticks(range(len(w_names)))
    axes[0].set_yticklabels(w_names, fontsize=8)
    axes[0].axvline(0, color='black', lw=0.8)
    axes[0].set_title('LIME Text\n(từ quan trọng)', fontsize=9, fontweight='bold')
    axes[0].set_xlabel('LIME weight', fontsize=8)
else:
    axes[0].text(0.5, 0.5, 'N/A', ha='center', va='center',
                 transform=axes[0].transAxes); axes[0].axis('off')

# Panels 2-4: image overlays
from PIL import Image as _PILImg3
for col_i, (key, title) in enumerate(
        [('positive_overlay','LIME Image (+)'),
         ('negative_overlay','LIME Image (−)'),
         ('combined_overlay','LIME Image (±)')]):
    ax = axes[col_i + 1]
    path = lime_img_paths.get(key)
    if path and os.path.exists(path):
        ax.imshow(np.array(_PILImg3.open(path)))
        ax.axis('off')
        ax.set_title(title, fontsize=9, fontweight='bold')
    else:
        ax.text(0.5, 0.5, 'N/A', ha='center', va='center',
                transform=ax.transAxes); ax.axis('off')

fig.suptitle(f'LIME Explanation — {SAMPLE_ID} (Overall Satisfaction)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
lime_4panel_path = f'{demo_dir}/{SAMPLE_ID}_lime_4panel.png'
fig.savefig(lime_4panel_path, dpi=DEFAULT_DPI, bbox_inches='tight', facecolor='white')
plt.show()
plt.close(fig)
print(f'[XAI] Đã lưu: {lime_4panel_path}')

## 1.8 · AI Agent — Báo cáo tổng hợp

> GPT-4o chỉ verbalize evidence đã tạo và chỉ được gọi khi `OPENAI_API_KEY`
> tồn tại trong environment/Colab Secrets. Thiếu key thì section được bỏ qua
> an toàn. AI Agent không được phép thay đổi dự đoán, bịa token/patch, hay
> đưa ra tuyên bố nhân quả không có căn cứ.

In [ ]:
agent_output = None
if AGENT_AVAILABLE:
    agent_config = AgentConfig(language='vi')
    if agent_config.api_key:
        print(f'[Agent] Đang chạy AI Agent cho {SAMPLE_ID} ...')
        agent = ExplanationAgent(agent_config)
        agent_output = run_safe(
            agent.explain_sample, step_name='AI_Agent',
            fallback=None,
            sample_id=SAMPLE_ID,
            review_text=sample['text'],
            predictions=pred_result['predictions'],
            xai_dir=XAI_DIR,
            ground_truth=pred_result['ground_truth'],
            case_type=CASE_TYPE,
            language='vi',
            mode='text_only',
            num_images=sample['num_real_images'],
            output_dir=agent_dir,
        )
        if agent_output:
            print(f"[Agent] ✅ Evidence completeness: {agent_output.get('evidence_completeness','N/A')}")
        else:
            print('[Agent] Không tạo được báo cáo.')
    else:
        print('[Agent] OPENAI_API_KEY không tìm thấy → bỏ qua.')
        print('  → Add OPENAI_API_KEY in Colab Secrets, then rerun this cell.')
else:
    print('[Agent] Module không khả dụng.')

## 1.9 · Kiểm tra artifact (chỉ cho `sample_0000`)

Kiểm tra tĩnh các file artifact đã được tạo trên đĩa cho từng phương pháp XAI.

In [ ]:
artifact_check = check_sample_artifacts(SAMPLE_ID, XAI_DIR)
print(f'\n{SAMPLE_ID} — Artifact completeness: {artifact_check.get("completeness", 0)*100:.0f}%')
for key, label in [
    ('gradcam', 'Grad-CAM'),
    ('attention', 'PhoBERT Attention'),
    ('cross_attention', 'Cross-Attention'),
    ('shap', 'SHAP'),
    ('lime', 'LIME'),
]:
    status = '✅' if artifact_check.get(key) else '⚠️  (chưa tạo — cần chạy notebook trên Colab)'
    print(f'  {status} {label}')
print(
    '\n[NOTE] Trạng thái trên phản ánh các file đã tồn tại trên đĩa tại thời '
    'điểm kiểm tra tĩnh này (notebook chưa được chạy). Chạy tuần tự trên '
    'Colab để tạo đầy đủ artifact.'
)

---
# 📊 PHẦN 2 — TÓM TẮT MỘT MẪU (sample_0000)

In [ ]:
# ── Tổng kết demo (một mẫu) ──────────────────────────────────────────────────
print('╔══════════════════════════════════════════════════════════╗')
print('║      TỔNG KẾT DEMO — sample_0000 (Bánh canh cua)          ║')
print('╠══════════════════════════════════════════════════════════╣')
print(f'║  Sample ID   : {SAMPLE_ID:<44s}║')
print(f'║  Test index  : {SAMPLE_INDEX:<44d}║')
print(f'║  Case type   : {CASE_TYPE:<44s}║')
print('╚══════════════════════════════════════════════════════════╝')

print('\nReview text:')
for line in textwrap.wrap(sample['text'], width=100):
    print(f'  {line}')

print(f'\nSố ảnh thực: {sample["num_real_images"]}')

print('\nDự đoán vs Thực tế:')
for name, disp in zip(TARGET_NAMES, DISPLAY_NAMES):
    p = pred_result['predictions'][name]
    g = pred_result['ground_truth'][name]
    e = pred_result['absolute_errors'][name]
    print(f'  {disp:<22s} pred={p:.2f}  gt={g:.2f}  AE={e:.3f}')
print(f'  Mean MAE: {pred_result["mean_mae"]:.3f}')

print('\nTrạng thái pipeline XAI (artifact đã kiểm tra tĩnh ở Phần 1.9):')
for key, label in [
    ('gradcam', 'Grad-CAM'),
    ('attention', 'PhoBERT Attention'),
    ('cross_attention', 'Cross-Attention'),
    ('shap', 'SHAP'),
    ('lime', 'LIME'),
]:
    status = '✅' if artifact_check.get(key) else '⚠️ '
    print(f'  {status} {label}')

agent_status = (
    '✅ đã tạo báo cáo' if agent_output
    else '⏭️ đã skip (thiếu OPENAI_API_KEY hoặc lỗi optional)'
)
print(f'  {agent_status} — AI Agent')

print(
    '\nCompleted the end-to-end static notebook design for sample_0000.\n'
    'Run the notebook sequentially on Google Colab to generate the actual outputs.'
)